In [1]:
import numpy as np
import pandas as pd
from sklearn.utils import resample
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from tabpfn import TabPFNClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# More Models

To determine whether the AdaBoost model provides sufficient performance, we evaluated more powerful learners, specifically TabPFN v3 and Random Forest. Consistent with the primary endpoint of the SURGE-Ahead study, **accuracy** was selected as the primary performance metric. We used **default hyperparameters**, as these generally yield good results, with TabPFN being particularly optimal out of the box.

## Load Data

In [2]:
data = pd.read_csv('./coc_data.csv')

In [3]:
mode = [
    'Geschlecht (m/w)', # Sex (m/w); m=1, w=0
    'Zu Hause mit Hilfe (Ja/Nein)', # Lives at home with help (Yes/No)
    'Pflegeheim (Ja/Nein)', # Nursing home (Yes/No)
    'Betreutes Wohnen (Ja/Nein)', # Assisted Living (Yes/No)
    'Zu Hause ohne Hilfe (Ja/Nein)', # Lives at home without help (Yes/No)
    'Bekommt im Alltag Hilfe (Ja/Nein)', # needs help in ADL (Yes/No)
    'Demenz (Ja/Nein)', # Dementia (Yes/No)
    'Transfusion (Ja/Nein)', # Transfusion (Yes/No)
]

median = [
    'Anzahl an Sozialkontakten (ordinal)', #Number of social contacts (ordinal)
    'Barthel-Index vor Aufnahme (Score, ordinal)', # Barthel Index before admission (Score, ordinal)
    'Barthel-Index bei Aufnahme (Score, ordinal)', # Barthel Index at admission (Score, ordinal)
    'Barthel-Index am post-OP Tag 1 (Score, ordinal)', # Barthel Index on post-OP day 1 (Score, ordinal)
    'Barthel Index am post-OP Tag 3 (Score, ordinal)', # Barthel Index on post-OP day 3 (Score, ordinal)
    'Barthel Index Subscore 2 (Aufsetzen und Umsetzen) am post-OP Tag 3 (Score, ordinal)', # Barthel Index Subscore 2 (Sitting and Transferring) on post-OP day 3 (Score, ordinal)
    'CHARMI vor Aufnahme (Score, ordinal)', # CHARMI before illness (Score, ordinal)
    'CHARMI am post-OP Tag 1 (Score, ordinal)', # CHARMI on post-OP day 1 (Score, ordinal)
    'CHARMI am post-OP Tag 3 (Score, ordinal)', # CHARMI on post-OP day 3 (Score, ordinal)
    'ISAR (Score, ordinal)', # ISAR (Score, ordinal)
    'Clinical Frailty Scale (Score, ordinal)', # Clinical Frailty Scale (Score, ordinal)
    'ASA (Score, ordinal)', # ASA (Score, ordinal)
    'Modifizierter Charlson Comorbidity Index (Score, ordinal)', # Modified Charlson Comorbidity Index (Score, ordinal)
    'Anzahl der Dauermedikamente (n)', # Number of long-term medications (n)
    'MOCA 5min (Score, ordinal)', # MOCA 5min (Score, ordinal)
    'Pflegegrad (Score, ordinal)', # Level of care (Score, ordinal)
]

mean = [
    'Alter bei OP (Jahre)', # Age at surgery (years)
    'OP Dauer (Minuten)', # Cut-to-suture time (minutes)
    'Liegedauer auf Intensivstation (Minuten)', # Length of stay in the ICU (minutes)
]

drop_list = [
    'Quelle', # Data Source (Observational Data from SURGE-Ahead (SA), or Trauma Register (ATZ)
    'Entlassungdestination (Kategorie)', # Discharge destination (category)
    'Kreuz-Validierung (Fold)', # Cross-Validation (CV; folds). In order to prevent information spillover, folds used in EDA are kept for CV.
]

data_type_tuple = (mode, median, mean)

In [4]:
X_SA = data.loc[data['Quelle'] == 'SA'].drop(drop_list[:2], axis=1) # Keep fold information in data.
X_ATZ = data.loc[data['Quelle'] == 'ATZ'].drop(drop_list, axis=1) # The data is used as additional training data within folds.
y_SA = data.loc[data['Quelle'] == 'SA', 'Entlassungdestination (Kategorie)'].values # save as numpy array
y_ATZ = data.loc[data['Quelle'] == 'ATZ', 'Entlassungdestination (Kategorie)'].values # save as numpy array

## Training and Evaluation

### Functions

In [5]:
def make_train_test(X_SA, X_ATZ, y_SA, y_ATZ, fold, additional_data):
    
    """
    Performs a train-test split based on cross-validation folds, with optional inclusion of additional data.

    This function divides the primary SURGE-Ahead data (`X_SA`, `y_SA`) into training and testing sets according to the cross-validation fold indicated in the 'Kreuz-Validierung (Fold)' column of `X_SA`. 
    One fold is used for testing, while the remaining folds constitute the training set. Optionally, data from a trauma register (`X_ATZ`, `y_ATZ`) can be appended to the training data.

    Args:
        X_SA (pd.DataFrame): Primary dataset with features, including a 'Kreuz-Validierung (Fold)' column for cross-validation.
        X_ATZ (pd.DataFrame): Additional dataset to potentially augment the training data.
        y_SA (pd.Series): Target variable for the primary dataset.
        y_ATZ (np.ndarray): Target variable for the additional dataset.
        fold (int): The current cross-validation fold to use as the test set.
        additional_data (bool):  Flag indicating whether to include the additional trauma register data in the training set.

    Returns:
        tuple:  A tuple containing the training features (X_train), testing features (X_test), training target (y_train), and testing target (y_test).
    """
    
    test_index = X_SA[X_SA['Kreuz-Validierung (Fold)'] == fold].index
    train_index = X_SA[X_SA['Kreuz-Validierung (Fold)'] != fold].index
    X_temp = X_SA.drop(['Kreuz-Validierung (Fold)'], axis=1)
    X_train_temp, X_test = X_temp.iloc[train_index], X_temp.iloc[test_index]
    y_train_temp, y_test = y_SA[train_index], y_SA[test_index]
    
    if additional_data: # if trauma register data should be incorporated into the training data
        X_train = pd.concat((X_train_temp, X_ATZ))
        y_train = np.concatenate((y_train_temp, y_ATZ)) 
    else:
        X_train = X_train_temp
        y_train = y_train_temp
        
    return X_train, X_test, y_train, y_test # Classic train test split.


def make_imputation(X_train, X_test, data_type_tuple=data_type_tuple):
    
    """
    Imputes missing values in the training and testing datasets using different strategies based on data type.

    This function addresses missing data in the training and testing sets by applying different imputation methods depending on the data type of the column. 
    It uses the 'most frequent' strategy for categorical features, the 'median' strategy for ordinal features, and the 'mean' strategy for continous features.

    Args:
        X_train (pd.DataFrame): Training dataset with potential missing values.
        X_test (pd.DataFrame): Testing dataset with potential missing values.
        data_type_tuple (tuple): A tuple containing three lists of column names: 
                                  the first list corresponds to columns to be imputed with the 'most frequent' strategy, 
                                  the second with the 'median', and the third with the 'mean'.

    Returns:
        tuple: A tuple containing the imputed training features (X_train) and testing features (X_test).
    """
    
    mode, median, mean = data_type_tuple # global variable
    
    imp_mode = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
    imp_mode.fit(X_train[mode])
    X_train[mode] = imp_mode.transform(X_train[mode])
    X_test[mode] = imp_mode.transform(X_test[mode])

    imp_median = SimpleImputer(missing_values=np.nan, strategy='median')
    imp_median.fit(X_train[median])
    X_train[median] = imp_median.transform(X_train[median])
    X_test[median] = imp_median.transform(X_test[median])
    
    imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
    imp_mean.fit(X_train[mean])
    X_train[mean] = imp_mean.transform(X_train[mean])
    X_test[mean] = imp_mean.transform(X_test[mean])

    return X_train, X_test

### TabPFN

In [7]:
cv_y_pred = []
cv_y_test = []

for fold in range(1, 6): # for each fold in the outer loop

    X_train, X_test, y_train, y_test = make_train_test(X_SA, X_ATZ, y_SA, y_ATZ, fold=fold, additional_data=True)
    X_train, X_test = make_imputation(X_train, X_test)
                    
    tabpfn = TabPFNClassifier(random_state=42)
    tabpfn.fit(X_train, y_train)
    
    cv_y_pred.append(tabpfn.predict(X_test))
    cv_y_test.append(y_test)

y_pred, y_true = np.concatenate(cv_y_pred), np.concatenate(cv_y_test)

accuracy_score(y_true, y_pred)

0.7810650887573964

### RandomForest

In [8]:
cv_y_pred = []
cv_y_test = []

for fold in range(1, 6): # for each fold in the outer loop

    X_train, X_test, y_train, y_test = make_train_test(X_SA, X_ATZ, y_SA, y_ATZ, fold=fold, additional_data=True)
    X_train, X_test = make_imputation(X_train, X_test)
                    
    rf = RandomForestClassifier(random_state=42)
    rf.fit(X_train, y_train)
    
    cv_y_pred.append(rf.predict(X_test))
    cv_y_test.append(y_test)

y_pred, y_true = np.concatenate(cv_y_pred), np.concatenate(cv_y_test)

accuracy_score(y_true, y_pred)

0.7988165680473372

# Confusion Matrix

To better inspect the model performance, we are generating a **confusion matrix**. This allows for a more detailed evaluation of the predicted classes compared to the actual labels.

## Load and Merge Data

For the confusion matrix, we need to merge the predictions generated during the AdaBoost model training and internal validation (`1_AdaBoost_OvO_Sigmoid_Training.ipynb`) with their corresponding ground-truth labels. As these data are currently stored in separate DataFrames, this consolidation is necessary to align the predictions with the actual labels.

In [9]:
df1 = pd.read_csv('./coc.csv') # Variables and Predictions from AdaBoost
df2 = data[data['Quelle'] == 'SA'].drop(['Quelle', 'Kreuz-Validierung (Fold)'], axis=1) # Labels

# Merge DataFrames
cols = df2.columns[(df2.isna().sum()==0)][:-1].to_list()
merged_df = pd.merge(df1, df2, on=cols, how='outer')

# Results only
results = merged_df[[
    'Entlassungdestination (Kategorie)',
    'coc_pred','coc_proba_class0',
    'coc_proba_class1',
    'coc_proba_class2',
    'coc_proba_class3'
]]

## Make Matrix

In [10]:
y_true = results['Entlassungdestination (Kategorie)'].values
y_pred = results['coc_pred'].values

# Generate the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Your defined label dictionary
coc_fig_dict = {
    0: 'Back Home',
    1: 'Acute Geriatric Care Unit',
    2: 'Rehabilitation',
    3: 'Nursing Home'
}

# Extract labels in the correct order (0, 1, 2, 3) using the dictionary values
labels = [coc_fig_dict[i] for i in sorted(coc_fig_dict.keys())]

# Convert the matrix into a pandas DataFrame for a table format
df_cm = pd.DataFrame(cm, index=labels, columns=labels)

# Rename axes for clarity
df_cm.index.name = 'Actual'
df_cm.columns.name = 'Predicted'

# Save as table
df_cm.to_csv('table2.csv')

df_cm

Predicted,Back Home,Acute Geriatric Care Unit,Rehabilitation,Nursing Home
Actual,,,,
Back Home,72,7,0,0
Acute Geriatric Care Unit,11,62,0,2
Rehabilitation,3,6,0,0
Nursing Home,0,2,0,4


## Calcuate PPV and NPV
Positive Predictive Value (PPV) and Negative Predictive Value (NPV) are calculated to translate model performance into clinical utility, providing the probability that a specific discharge recommendation is correct (PPV) or that a patient truly does not require a specific care level (NPV).

In [11]:
metrics = []
labels = df_cm.index

for i in range(len(labels)):
    # True Positives: Diagonal element
    tp = df_cm.iloc[i, i]

    # False Positives: Sum of the column minus TP
    fp = df_cm.iloc[:, i].sum() - tp

    # False Negatives: Sum of the row minus TP
    fn = df_cm.iloc[i, :].sum() - tp

    # True Negatives: Total sum minus (TP + FP + FN)
    tn = df_cm.values.sum() - (tp + fp + fn)

    # PPV = TP / (TP + FP)
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0

    # NPV = TN / (TN + FN)
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    metrics.append({
        'Destination': labels[i],
        'PPV': ppv,
        'NPV': npv
    })

# Convert to DataFrame for a clean overview
df_performance = pd.DataFrame(metrics)
df_performance.set_index('Destination', inplace=True)

df_performance.round(2)

,PPV,NPV
Destination,,
Back Home,0.84,0.92
Acute Geriatric Care Unit,0.81,0.86
Rehabilitation,0.00,0.95
Nursing Home,0.67,0.99
